# 🇸🇬 CATI Training — Context-Aware Traffic Intelligence

**Platform:** Google Colab (T4 / L4 / A100 GPU)

This notebook executes the full CATI training pipeline:

| Step | Cell | Notes |
|------|------|-------|
| 0. Setup | Cells 1–3 | Mount Drive, clone repo, install deps |
| 1. Verify data | Cell 4 | Check raw images collected by collect_data.ipynb |
| 2. Hook verification | Cell 5 | Confirm forward hooks capture correct layers |
| 3. Feature extraction | Cell 6 | Run once; saves .pt + .json to Drive |
| 4. Phase 1 training | Cell 7 | Context modules only; fast, VRAM-efficient |
| 5. Evaluate Phase 1 | Cell 8 | Stratified validation summary |
| 6. Phase 2 training | Cell 9 | End-to-end fine-tuning (optional, more VRAM) |

---
**Storage estimate:** ~2.8 MB per image (FP16).  
Set `MAX_FEATURE_SAMPLES` below to match your Drive free space.

In [ ]:
# ============================================================
# Cell 1: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import shutil
total, used, free = shutil.disk_usage('/content/drive')
free_gb = free / (2**30)
print(f'Drive free: {free_gb:.1f} GB')
if free_gb < 5:
    print('⚠️  WARNING: Less than 5 GB free. Feature extraction may fail.')

In [ ]:
# ============================================================
# Cell 2: Clone / pull repo
# ============================================================
import os

REPO_URL = 'https://github.com/Suhxs-Reddy/sg-smart-city-analytics.git'
REPO_DIR = '/content/sg-smart-city-analytics'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull --quiet
    print('✅ Repo updated')
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}
    print('✅ Repo cloned')

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ============================================================
# Cell 3: Install dependencies & verify GPU
# ============================================================
!pip install -q ultralytics torch torchvision click pyyaml wandb

import torch
import sys
sys.path.insert(0, REPO_DIR)

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU:      {gpu.name} ({gpu.total_memory / 2**30:.1f} GB VRAM)')
else:
    print('⚠️  No GPU detected — feature extraction will be very slow on CPU.')

In [ ]:
# ============================================================
# Cell 4: Verify collected raw data
# ============================================================
from pathlib import Path

RAW_DIR     = '/content/drive/MyDrive/sg_smart_city/data/raw'
FEATURE_DIR = '/content/drive/MyDrive/sg_smart_city/data/features'
MODEL_DIR   = '/content/drive/MyDrive/sg_smart_city/models'

# ---- Adjust this to stay within Drive quota ----
# Storage: ~2.8 MB/image (FP16).  1000 images ≈ 2.8 GB.
MAX_FEATURE_SAMPLES = 2000  # Set to None to process everything
# ------------------------------------------------

raw_path = Path(RAW_DIR)
if not raw_path.exists():
    print(f'❌ Raw data not found at {RAW_DIR}')
    print('   Run collect_data.ipynb first to gather images from LTA cameras.')
else:
    jpg_count   = len(list(raw_path.rglob('*.jpg')))
    jsonl_count = len(list(raw_path.rglob('*.jsonl')))
    total_mb    = sum(f.stat().st_size for f in raw_path.rglob('*') if f.is_file()) / 1e6
    cameras     = {d.name for d in raw_path.rglob('*') if d.is_dir() and d.name.isdigit()}

    print(f'📊 Raw data summary')
    print(f'   Images:       {jpg_count:,}')
    print(f'   Metadata:     {jsonl_count:,} JSONL files')
    print(f'   Total size:   {total_mb:.0f} MB')
    print(f'   Cameras:      {len(cameras)}')
    print()

    if jpg_count == 0:
        print('❌ No images found. Run collect_data.ipynb first.')
    elif jpg_count < 500:
        print(f'⚠️  Only {jpg_count} images — run data collection longer for better training.')
    else:
        print(f'✅ Enough data for initial training.')

    print(f'\nFeature extraction will process up to {MAX_FEATURE_SAMPLES} samples.')
    est_gb = (MAX_FEATURE_SAMPLES or jpg_count) * 2.8 / 1024
    print(f'Estimated storage needed: ~{est_gb:.1f} GB')

In [ ]:
# ============================================================
# Cell 5: Verify forward hooks (MUST run before extraction)
# ============================================================
# This confirms that our hooks capture features at the right
# backbone layers [4, 6, 9] which correspond to P3, P4, P5.
# Expected shapes for YOLOv11s with img_size=640:
#   Layer 4 (P3): (1, 128, 80, 80)
#   Layer 6 (P4): (1, 256, 40, 40)
#   Layer 9 (P5): (1, 512, 20, 20)
# If shapes are different, update BACKBONE_LAYERS in feature_extractor.py.

from src.training.feature_extractor import YOLOFeatureExtractor

print('Loading YOLOv11s and verifying hook layer shapes...')
yfe = YOLOFeatureExtractor(model_path='yolo11s.pt', device='auto')
shapes = yfe.verify_hooks(img_size=640)
yfe.remove_hooks()

EXPECTED = {
    4: (1, 128, 80, 80),
    6: (1, 256, 40, 40),
    9: (1, 512, 20, 20),
}
all_ok = True
for layer_idx, expected_shape in EXPECTED.items():
    actual = shapes.get(layer_idx)
    ok = actual == expected_shape
    if not ok:
        all_ok = False
    status = '✅' if ok else '❌'
    print(f'  {status} Layer {layer_idx}: expected {expected_shape}, got {actual}')

if all_ok:
    print('\n✅ Hook verification passed — proceed to feature extraction.')
else:
    print('\n❌ Shape mismatch! Update BACKBONE_LAYERS in feature_extractor.py before extracting.')
    print('   Run: for i, m in enumerate(yfe.yolo.model.model): print(i, m.__class__.__name__)')

In [ ]:
# ============================================================
# Cell 6: Feature extraction
# ============================================================
# Run this ONCE. Output is saved to FEATURE_DIR on Drive.
# If you interrupt and resume, already-processed files are NOT
# re-extracted (check counts to see what's there already).

import os
from pathlib import Path
from src.training.feature_extractor import FeatureExtractor
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

# Check if features already exist
feat_path = Path(FEATURE_DIR)
existing = {split: len(list((feat_path / split).glob('*.json')))
            for split in ['train', 'val', 'test']
            if (feat_path / split).exists()}
if existing:
    total_existing = sum(existing.values())
    print(f'Found existing features: {existing} (total={total_existing})')
    print('If you want to re-extract, delete the feature directory first.')
    SKIP_EXTRACTION = total_existing >= (MAX_FEATURE_SAMPLES or 100)
else:
    SKIP_EXTRACTION = False

if SKIP_EXTRACTION:
    print('✅ Skipping extraction — enough features already on Drive.')
else:
    print(f'Starting feature extraction (max_samples={MAX_FEATURE_SAMPLES})...')
    print('This runs YOLO forward passes on all raw images — expect 10–60 min on T4.')
    print()

    extractor = FeatureExtractor(model_path='yolo11s.pt', device='auto')
    stats = extractor.extract_all(
        raw_dir=RAW_DIR,
        output_dir=FEATURE_DIR,
        max_samples=MAX_FEATURE_SAMPLES,
    )
    extractor.cleanup()

    print(f'\n✅ Feature extraction complete: {stats}')

    # Verify output
    for split in ['train', 'val', 'test']:
        split_dir = Path(FEATURE_DIR) / split
        if split_dir.exists():
            jsons = len(list(split_dir.glob('*.json')))
            pts   = len(list(split_dir.glob('*.pt')))
            print(f'  {split}: {jsons} json, {pts} pt  ({"✅" if jsons == pts else "⚠️ mismatch"})')

In [ ]:
# ============================================================
# Cell 7: Phase 1 Training — Context Modules Only
# ============================================================
# Trains ContextEncoder + FiLM layers using cached backbone features.
# The YOLO backbone is NOT loaded — only the 200K CATI parameters train.
# Expected: ~15–30 min on T4 GPU for 50 epochs on 2000 samples.

import os
import logging
from pathlib import Path
from torch.utils.data import DataLoader

from src.models.cati_detector import CATIConfig
from src.training.train_cati import CATIDataset, CATITrainer, cati_collate

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

os.makedirs(MODEL_DIR, exist_ok=True)

# ---- Hyperparameters ----
PHASE1_EPOCHS      = 50
PHASE1_LR          = 1e-3
PHASE1_BATCH_SIZE  = 32      # Can be larger in Phase 1 (no YOLO in memory)
PHASE1_WARMUP      = 5
PATIENCE           = 15
RESUME_FROM        = None    # Set to a .pt path to resume: f'{MODEL_DIR}/cati_latest.pt'
# -------------------------

# (Optional) WandB experiment tracking
try:
    import wandb
    wandb.login(anonymous='allow')
    wandb.init(project='sg-smart-city-cati', name='phase1', config={
        'phase': 1, 'epochs': PHASE1_EPOCHS, 'lr': PHASE1_LR, 'batch_size': PHASE1_BATCH_SIZE
    })
    USE_WANDB = True
    print('✅ WandB initialized')
except Exception:
    USE_WANDB = False
    print('ℹ️  WandB not available — training without experiment tracking.')

# Build datasets
feat_path = Path(FEATURE_DIR)
train_ds = CATIDataset(str(feat_path / 'train'))
val_ds   = CATIDataset(str(feat_path / 'val'))

if len(train_ds) == 0:
    raise RuntimeError('Training dataset is empty! Run Cell 6 (feature extraction) first.')

train_loader = DataLoader(
    train_ds, batch_size=PHASE1_BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, collate_fn=cati_collate
)
val_loader = DataLoader(
    val_ds, batch_size=PHASE1_BATCH_SIZE, shuffle=False,
    num_workers=2, collate_fn=cati_collate
)

# Initialize trainer
config = CATIConfig(
    num_cameras=90,
    context_dim=64,
    camera_embed_dim=16,
    use_gps_encoding=True,
    use_attention=True,
    ema_decay=0.9999,
)
trainer = CATITrainer(
    config=config,
    learning_rate=PHASE1_LR,
    weight_decay=1e-4,
    warmup_epochs=PHASE1_WARMUP,
    use_amp=True,
    grad_accum_steps=2,
)

if RESUME_FROM:
    trainer.load_checkpoint(RESUME_FROM)
    print(f'Resumed from {RESUME_FROM}')

print(f'Phase 1 training: {len(train_ds)} train, {len(val_ds)} val')
print(f'Epochs: {PHASE1_EPOCHS} | LR: {PHASE1_LR} | Batch: {PHASE1_BATCH_SIZE}')
print()

# Train
result = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=PHASE1_EPOCHS,
    save_dir=MODEL_DIR,
    patience=PATIENCE,
)

if USE_WANDB:
    wandb.finish()

print(f'\n✅ Phase 1 complete! Best val_loss: {result["best_val_loss"]:.4f}')
print(f'Checkpoint saved to: {MODEL_DIR}/cati_best.pt')

In [ ]:
# ============================================================
# Cell 8: Evaluate Phase 1 — Stratified Validation Summary
# ============================================================
# Shows per-condition losses to identify model weaknesses before Phase 2.

import json
from pathlib import Path

best_ckpt = Path(MODEL_DIR) / 'cati_best.pt'
if not best_ckpt.exists():
    print('No checkpoint found — run Cell 7 first.')
else:
    import torch
    ckpt = torch.load(str(best_ckpt), map_location='cpu', weights_only=False)
    history = ckpt.get('training_history', [])

    print(f'Best checkpoint: epoch {ckpt["epoch"]} | val_loss = {ckpt["val_loss"]:.4f}')
    print(f'Total epochs trained: {len(history)}')
    print()

    # Plot training curve
    try:
        import matplotlib.pyplot as plt
        train_losses = [h['train_loss'] for h in history]
        epochs = list(range(1, len(train_losses) + 1))

        plt.figure(figsize=(10, 4))
        plt.plot(epochs, train_losses, label='Train Loss')
        plt.xlabel('Epoch')
        plt.ylabel('MSE Loss')
        plt.title('CATI Phase 1 — Modulation Loss')
        plt.legend()
        plt.tight_layout()
        plt.show()
    except Exception:
        pass

    # Run stratified validation
    from src.models.cati_detector import CATIConfig, CATIDetector
    from src.training.train_cati import CATIDataset, CATITrainer, cati_collate
    from torch.utils.data import DataLoader

    feat_path = Path(FEATURE_DIR)
    val_ds = CATIDataset(str(feat_path / 'val'))
    val_loader = DataLoader(val_ds, batch_size=16, collate_fn=cati_collate)

    config = CATIConfig(**{k: v for k, v in ckpt['config'].items()
                           if hasattr(CATIConfig, k)})
    trainer = CATITrainer(config=config, device='auto')
    trainer.cati.load_state_dict(ckpt['model_state_dict'])

    val_metrics = trainer.validate(val_loader)
    print('Stratified validation metrics:')
    for k, v in sorted(val_metrics.items()):
        print(f'  {k:<45} {v:.4f}')

    print()
    print('Key insights to look for:')
    print('  - High loss on weather/heavy_rain → model needs more rain samples')
    print('  - High loss on resolution/low     → model struggles with 240p cameras')
    print('  - High loss on time_of_day/night  → more night-time data needed')

In [ ]:
# ============================================================
# Cell 9: Phase 2 — End-to-End Fine-tuning (Optional)
# ============================================================
# Loads the full YOLOv11s + CATI module and jointly fine-tunes them.
# Requires the YOLO backbone in GPU memory — needs ≥12 GB VRAM (T4 OK).
#
# ⚠️  This cell requires Phase 1 to be complete and a valid cati_best.pt
#     in MODEL_DIR. Phase 2 is more VRAM-intensive — reduce batch size
#     if you get OOM errors.
#
# Note: Phase 2 uses the YOLO detection head for loss computation.
# The backbone hooks from CATIBackboneWrapper wire the FiLM conditioning
# into the live YOLO forward pass. This is still TODO in cati_detector.py
# (see CATIBackboneWrapper.predict) — implement full hook integration there
# before running Phase 2 in production.

print('Phase 2 end-to-end fine-tuning is not yet implemented in this notebook.')
print()
print('Prerequisites before running Phase 2:')
print('  1. Complete Phase 1 (Cell 7) and confirm good val_loss < 0.05')
print('  2. Implement CATIBackboneWrapper hook wiring in src/models/cati_detector.py')
print('     - Register forward hooks on yolo.model.model[4, 6, 9]')
print('     - In each hook: apply self.cati film conditioning to captured features')
print('     - Replace hook output tensor in-place so YOLO detection head')
print('       receives the modulated features')
print('  3. Prepare YOLO-format labels (from detections in .json files)')
print('  4. Run: python -m src.training.train_cati --phase 2 --data-dir data/features')